In [ ]:
# Default Packges
import numpy as np
import pandas as pd
import json, csv
import re

# Used to pull song lyrics
import lyricsgenius

# Model
from transformers import (
    LongformerTokenizer, 
    LongformerForSequenceClassification,
    TrainingArguments, 
    Trainer
)

import torch
from transformers import pipeline
from datasets import Dataset
import ast
from torch.utils.data import Dataset

from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, roc_auc_score


# Hugging Face
from huggingface_hub import login

login(token="#")

# Functions

In [7]:
"""Function cleans string and leaves only the song lyrics we want"""
def clean_lyrics(raw_lyrics):
    #Get rid of information shoved at beginning of string leaving only the lyrics of the song
    starter = ']' #This always finds the first closed bracket which is always the closing bracket for '[Intro]'
    position = raw_lyrics.find(starter)
    lyrics = raw_lyrics[position+1:]

    # Remove section headers like [Chorus], [Verse 1], etc.
    lyrics = re.sub(r"\[.*?\]", "", lyrics)

    # Remove extra whitespace, newlines
    lyrics = re.sub(r"\n+", " ", lyrics)
    lyrics = re.sub(r"\s+", " ", lyrics)

    # Optionally remove punctuation
    lyrics = re.sub(r"[^\w\s']", "", lyrics)

    # Lowercase
    lyrics = lyrics.lower()

    return lyrics

"""Function searchs song and returns string containing only the lyrics"""
def song_search(song,artist):
    genius = lyricsgenius.Genius("rPVs-pFT7GhBfTxhpu-ISnNCcvCsbRt8wIwhkkEXovrADgSBQfkndQW7Ge22R5Ts", timeout=60)
    song = genius.search_song(song, artist)
    lyrics = song.lyrics
    
    cleaned_lyrics = clean_lyrics(lyrics)
    return cleaned_lyrics

In [ ]:
# import os
# os.environ["CUDA_LAUNCH_BLOCKING"] = "1"
# os.environ["TOKENIZERS_PARALLELISM"] = "false"

# # Limit GPU memory growth so crashes don't corrupt the context
# os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "max_split_size_mb:512"

In [8]:
# load in lyrics
lyrics_df = pd.read_csv('lyrics_list.txt', delimiter=',',header=None)
lyrics_df.columns = ['lyrics','Nan']
lyrics_df = lyrics_df.drop(labels='Nan',axis=1)

In [9]:
lyrics_df

,lyrics
0,hey jude don't make it bad take a sad song and...
1,when i find myself in times of trouble mother...
2,yesterday all my troubles seemed so far away ...
3,shoot me shoot me shoot me shoot me here come...
4,it's been a hard day's night and i've been wo...
...,...
904,you took me back but you shouldn't have now i...
905,i've been waiting i've been waiting for this ...
906,time is never worth my time blue shine bleeds...
907,i could run i could cry i could plead i could...


In [6]:
# model_name = "allenai/longformer-base-4096"
# tokenizer = LongformerTokenizer.from_pretrained(model_name)
# model = LongformerForSequenceClassification.from_pretrained(
#     model_name,
#     num_labels=10,          # set to your number of theme categories
#     problem_type="multi_label_classification"
# )

In [7]:
# def encode_lyrics(lyrics: str):
#     return tokenizer(
#         lyrics,
#         max_length=4096,
#         padding="max_length",
#         truncation=True,
#         return_tensors="pt"
#     )


# # Global attention on [CLS] token (important for Longformer classification)
# def forward_pass(lyrics: str):
#     inputs = encode_lyrics(lyrics)
#     # Longformer needs global_attention_mask — put global attention on [CLS]
#     global_attention_mask = torch.zeros_like(inputs["input_ids"])
#     global_attention_mask[:, 0] = 1   # [CLS] token gets global attention
#     inputs["global_attention_mask"] = global_attention_mask

#     with torch.no_grad():
#         outputs = model(**inputs)
#     logits = outputs.logits
#     probs = torch.sigmoid(logits)   # multi-label: sigmoid, not softmax
#     return probs

### Fine Tuning

In [8]:

# # Assume your data has columns: "lyrics", "themes" (list of labels)
# df = pd.read_csv("your_lyrics_dataset.csv")
# dataset = Dataset.from_pandas(df)

# training_args = TrainingArguments(
#     output_dir="./lyric-theme-model",
#     num_train_epochs=5,
#     per_device_train_batch_size=2,   # Longformer is memory-heavy; keep small
#     gradient_accumulation_steps=8,   # effective batch size = 16
#     warmup_ratio=0.1,
#     weight_decay=0.01,
#     logging_steps=50,
#     evaluation_strategy="epoch",
#     save_strategy="epoch",
#     load_best_model_at_end=True,
#     fp16=True,                       # use if your GPU supports it
# )

# trainer = Trainer(
#     model=model,
#     args=training_args,
#     train_dataset=dataset["train"],
#     eval_dataset=dataset["test"],
# )
# trainer.train()

# Assigning classifications to lyrics list

shouldnt need to run this any more

In [9]:
# facebook/bart-large-mnli is the best zero-shot classifier available
# It uses NLI (natural language inference) to score each label against your text
zero_shot = pipeline(
    "zero-shot-classification",
    model="facebook/bart-large-mnli",
    device=0  # GPU 0
)

THEMES = [
    "love and romance", "heartbreak and loss", "identity and self-discovery",
    "rebellion and resistance", "nostalgia and memory", "depression and mental health",
    "social commentary", "celebration and joy", "spirituality and faith",
    "ambition and success", "loneliness and isolation", "death and mortality",
]

THRESHOLD = 0.20  # labels scoring above this are assigned — tune after inspection

lyrics_list = lyrics_df["lyrics"].tolist()

# Process in batches — GPU stays busy between items
results = zero_shot(
    lyrics_list,
    candidate_labels=THEMES,
    multi_label=True,
    batch_size=8        # increase to 16 if you have VRAM headroom
)

# Parse results into labels and scores
assigned_labels = []
all_scores = []

for result in results:
    score_map = dict(zip(result["labels"], result["scores"]))
    assigned = [t for t, s in score_map.items() if s >= THRESHOLD]
    if not assigned:
        assigned = [max(score_map, key=score_map.get)]
    assigned_labels.append(assigned)
    all_scores.append(score_map)

lyrics_df["themes"] = assigned_labels
lyrics_df["theme_scores"] = all_scores
lyrics_df.to_json("lyrics_labeled.json", orient="records", lines=True)
print("Done. Labeled dataset saved.")

# def label_lyrics(lyrics: str) -> list[str]:
#     # Truncate to 1024 tokens — bart-large-mnli's limit
#     # For very long lyrics, we chunk and average scores
#     words = lyrics.split()
#     chunk_size = 400  # safe word count per chunk
#     chunks = [" ".join(words[i:i+chunk_size]) for i in range(0, len(words), chunk_size)]

#     # Accumulate scores across chunks
#     score_map = {theme: 0.0 for theme in THEMES}
#     for chunk in chunks:
#         result = zero_shot(chunk, THEMES, multi_label=True)
#         for label, score in zip(result["labels"], result["scores"]):
#             score_map[label] += score

#     # Average across chunks
#     n = len(chunks)
#     avg_scores = {t: score_map[t] / n for t in THEMES}

#     # Assign labels above threshold
#     assigned = [t for t, s in avg_scores.items() if s >= THRESHOLD]

#     # Always assign at least the top label to avoid empty rows
#     if not assigned:
#         assigned = [max(avg_scores, key=avg_scores.get)]

#     return assigned, avg_scores  # return scores too so you can review them


# # Run over your dataset


# assigned_labels = []
# all_scores = []

# for i, row in lyrics_df.iterrows():
#     labels, scores = label_lyrics(row["lyrics"])
#     assigned_labels.append(labels)
#     all_scores.append(scores)
#     if i % 50 == 0:
#         print(f"Processed {i}/{len(lyrics_df)} — last labels: {labels}")

# lyrics_df["themes"] = assigned_labels
# lyrics_df["theme_scores"] = all_scores
# lyrics_df.to_csv("lyrics_labeled.csv", index=False)
# print("Done. Labeled dataset saved.")

Device set to use cuda:0


Done. Labeled dataset saved.


In [ ]:
# Can probably delete

# import gc
# import torch

# gc.collect()
# torch.cuda.empty_cache()

# print(f"VRAM free:  {torch.cuda.mem_get_info()[0] / 1e9:.2f} GB")
# print(f"VRAM total: {torch.cuda.mem_get_info()[1] / 1e9:.2f} GB")

VRAM free:  5.72 GB
VRAM total: 8.59 GB


In [13]:
# ── Config ──────────────────────────────────────────────────────────────────
MODEL_NAME  = "allenai/longformer-base-4096"
MAX_LENGTH  = 2048   # 16GB VRAM handles this comfortably with batch size 2
BATCH_SIZE  = 2
GRAD_ACCUM  = 8      # effective batch size = 16
EPOCHS      = 6
LR          = 2e-5

THEMES = [
    "love and romance", "heartbreak and loss", "identity and self-discovery",
    "rebellion and resistance", "nostalgia and memory", "depression and mental health",
    "social commentary", "celebration and joy", "spirituality and faith",
    "ambition and success", "loneliness and isolation", "death and mortality",
]
NUM_LABELS = len(THEMES)
THEME_TO_IDX = {t: i for i, t in enumerate(THEMES)}

# ── Dataset class ────────────────────────────────────────────────────────────
class LyricsDataset(Dataset):
    def __init__(self, lyrics_list, labels_list, tokenizer):
        self.encodings = []
        self.labels = []

        for lyrics, themes in zip(lyrics_list, labels_list):
            enc = tokenizer(
                lyrics,
                max_length=MAX_LENGTH,
                padding="max_length",
                truncation=True,
                return_tensors="pt",
            )
            # Global attention on [CLS] — critical for Longformer classification
            global_attn = torch.zeros(MAX_LENGTH, dtype=torch.long)
            global_attn[0] = 1

            self.encodings.append({
                "input_ids":            enc["input_ids"].squeeze(),
                "attention_mask":       enc["attention_mask"].squeeze(),
                "global_attention_mask": global_attn,
            })

            # Build multi-hot label vector
            label_vec = torch.zeros(NUM_LABELS)
            for theme in themes:
                if theme in THEME_TO_IDX:
                    label_vec[THEME_TO_IDX[theme]] = 1.0
            self.labels.append(label_vec)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return {**self.encodings[idx], "labels": self.labels[idx]}

## This wsa for training the model (keep but no longer need to run since I have saved all of my model's weights in hugging face)

# # ── Load & split data ────────────────────────────────────────────────────────
# #lyrics_df = pd.read_csv("lyrics_labeled.csv")
# #lyrics_df["themes"] = lyrics_df["themes"].apply(ast.literal_eval)  # string → list

# train_df, val_df = train_test_split(lyrics_df, test_size=0.15, random_state=42)

# tokenizer = LongformerTokenizer.from_pretrained(MODEL_NAME)

# train_dataset = LyricsDataset(train_df["lyrics"].tolist(), train_df["themes"].tolist(), tokenizer)
# val_dataset   = LyricsDataset(val_df["lyrics"].tolist(),   val_df["themes"].tolist(),   tokenizer)


# # ── Model ─────────────────────────────────────────────────────────────────────
# model = LongformerForSequenceClassification.from_pretrained(
#     MODEL_NAME,
#     num_labels=NUM_LABELS,
#     problem_type="multi_label_classification",
# )


# # ── Metrics ───────────────────────────────────────────────────────────────────
# def compute_metrics(eval_pred):
#     logits, labels = eval_pred
#     probs = torch.sigmoid(torch.tensor(logits)).numpy()
#     preds = (probs >= 0.5).astype(int)
#     return {
#         "f1_micro":  f1_score(labels, preds, average="micro",  zero_division=0),
#         "f1_macro":  f1_score(labels, preds, average="macro",  zero_division=0),
#         "roc_auc":   roc_auc_score(labels, probs, average="macro"),
#     }


# ── Training ──────────────────────────────────────────────────────────────────
# training_args = TrainingArguments(
#     output_dir="./lyric-theme-longformer",
#     num_train_epochs=EPOCHS,
#     per_device_train_batch_size=1,
#     per_device_eval_batch_size=1,
#     gradient_accumulation_steps=16,
#     learning_rate=LR,
#     warmup_ratio=0.1,
#     weight_decay=0.01,
#     eval_strategy="epoch",
#     save_strategy="epoch",
#     load_best_model_at_end=True,
#     metric_for_best_model="f1_micro",
#     fp16=False,
#     bf16=True,
#     gradient_checkpointing=True,
#     dataloader_num_workers=0,
#     no_cuda=False,
#     logging_steps=25,
#     report_to="none",
# )

# trainer = Trainer(
#     model=model,
#     args=training_args,
#     train_dataset=train_dataset,
#     eval_dataset=val_dataset,
#     compute_metrics=compute_metrics,
# )

# trainer.train()
# trainer.save_model("./lyric-theme-longformer/best")
# tokenizer.save_pretrained("./lyric-theme-longformer/best")
# print("Training complete.")

In [ ]:
# This code allowed me to pull the results from my trained model

# Load from the best checkpoint that was saved during training
LOCAL_PATH = "./lyric-theme-longformer/best"

model = LongformerForSequenceClassification.from_pretrained(LOCAL_PATH)
tokenizer = LongformerTokenizer.from_pretrained(LOCAL_PATH)

print("Model loaded successfully")

Model loaded successfully


In [ ]:
# This code is how I saved my model weights to hugging face
HF_REPO = "gpecorino/lyric-theme-longformer"

model.push_to_hub(HF_REPO)
tokenizer.push_to_hub(HF_REPO)
print(f"Model saved to https://huggingface.co/{HF_REPO}")

Processing Files (1 / 1): 100%|██████████|  595MB /  595MB, 17.2MB/s  
New Data Upload: 100%|██████████|  595MB /  595MB, 17.2MB/s  
c:\Users\Monado\AppData\Local\Programs\Python\Python311\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Monado\.cache\huggingface\hub\models--gpecorino--lyric-theme-longformer. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable

Model saved to https://huggingface.co/gpecorino/lyric-theme-longformer


In [10]:
# This code is how I pull my saved weights from hugging face
HF_REPO = "gpecorino/lyric-theme-longformer"

model = LongformerForSequenceClassification.from_pretrained(HF_REPO)
tokenizer = LongformerTokenizer.from_pretrained(HF_REPO)

# Loading in new songs lyrics to test model with

In [16]:
cleaned_lyrics = song_search('Risk it All','Bruno Mars')
print(cleaned_lyrics)

Searching for "Risk it All" by Bruno Mars...
Done.
 for just the chance to win your heart you could set the bar beyond the stars i'll do anything anything you ask me to say you want the moon watch me learn to fly ain't no mountain you could point to i wouldn't climb it's crazy but it's true there's nothing i won't do i'd risk it all for you to hold your hand and call you mine i'm tryna be your man 'til the end of time oh i'll do anything anything you ask me to i would run through a fire just to be by your side if your heart's on the line you could take mine it's crazy but it's true there's nothing i won't do i'd risk it all for you i would swim across the sea just to show you sacrifice my life just to hold you i could go on and on to prove that you belong here in my arms say you want the moon watch me learn to fly ain't no mountain you could point to i wouldn't climb it's crazy but it's true there's nothing i won't do i'd risk it all for you it's crazy but it's true there's nothing i w

# Predicting Themese with model

In [17]:
def predict_themes(lyrics: str, threshold: float = 0.5) -> dict:
    model.eval()
    inputs = tokenizer(
        lyrics,
        max_length=MAX_LENGTH,
        padding="max_length",
        truncation=True,
        return_tensors="pt",
    ).to(model.device)

    global_attn = torch.zeros_like(inputs["input_ids"])
    global_attn[:, 0] = 1
    inputs["global_attention_mask"] = global_attn

    with torch.no_grad():
        logits = model(**inputs).logits

    probs = torch.sigmoid(logits).squeeze().cpu().numpy()
    results = {THEMES[i]: round(float(probs[i]), 3) for i in range(NUM_LABELS)}

    predicted = {t: s for t, s in results.items() if s >= threshold}
    return dict(sorted(predicted.items(), key=lambda x: -x[1]))

# Example
themes = predict_themes(cleaned_lyrics)
print(themes)
# → {'heartbreak and loss': 0.91, 'nostalgia and memory': 0.84, 'loneliness and isolation': 0.72}

{'identity and self-discovery': 0.95, 'social commentary': 0.943, 'nostalgia and memory': 0.929, 'love and romance': 0.924, 'celebration and joy': 0.919, 'spirituality and faith': 0.838, 'rebellion and resistance': 0.833, 'depression and mental health': 0.775, 'ambition and success': 0.754, 'heartbreak and loss': 0.726, 'loneliness and isolation': 0.654, 'death and mortality': 0.65}


In [15]:
print(len(themes))

12
